# Day 63: Lab - Sales Forecasting Mini-Project

## Project Goal
This lab combines everything from the ML-features approach (Day 62) into a small end-to-end sales forecasting project: build lag/rolling/calendar features from raw daily sales, split chronologically, train a gradient boosting model, and evaluate it both numerically (MAE) and visually (predicted vs actual). This mirrors how forecasting is often done in industry when you have external drivers you'd want to add later (promotions, holidays, pricing) that a classical model like ARIMA can't easily incorporate but a tree-based regressor can. Keep in mind the same caveat from Day 62 applies here too: tree-based models like gradient boosting can't extrapolate a trend beyond the range of values they saw in training, so on a sales series with a strong upward trend (like the one below), predictions on the test period will tend to undershoot as the trend continues past the training range — a useful real limitation to watch for, not just a demo quirk.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(4)
dates = pd.date_range('2023-01-01', periods=180, freq='D')
# Simulated daily sales: upward trend (+1.2/day) + weekly-ish seasonal wave + random noise
sales = 300 + np.arange(180) * 1.2 + 40 * np.sin(np.arange(180) / 7) + np.random.normal(0, 15, 180)
df = pd.DataFrame({'date': dates, 'sales': sales})

## Step 1: Feature Engineering
As in Day 62, we translate temporal structure into explicit numeric features an ML regressor can use directly. Lags of 1, 7, and 14 days give the model recent-day, one-week-ago, and two-weeks-ago sales values (useful given the weekly-scale oscillation in the data); a 7-day rolling mean (computed on past values only, via `shift(1)` first) gives a smoothed sense of the recent sales level; and `day_of_week` captures any recurring weekly pattern (e.g. weekend spikes). As before, the earliest rows lack enough history to compute these features, so `dropna()` removes them before modeling.

In [ ]:
# Lag features: sales from 1, 7, and 14 days ago
for lag in [1, 7, 14]:
    df[f'lag_{lag}'] = df['sales'].shift(lag)
# 7-day rolling average of past sales (shifted so today's value isn't leaked into its own feature)
df['rolling_mean_7'] = df['sales'].shift(1).rolling(7).mean()
# Day of week (0=Mon..6=Sun) to capture weekly patterns
df['day_of_week'] = df['date'].dt.dayofweek
# Drop the early rows that don't have enough history for lag/rolling features
df = df.dropna().reset_index(drop=True)

## Step 2: Time-Based Train-Test Split
As with Day 62, shuffling and randomly splitting a time series before train/test would leak future information into training (since lag features derived from later rows would end up alongside earlier rows in the training set) and would make test performance look unrealistically good. Instead we cut the data at a point in time: the earliest 85% of days form the training set, and the most recent 15% form the test set, simulating the real task of forecasting days you haven't seen yet using only what happened before them.

In [ ]:
# Chronological split: earliest 85% of days for training, most recent 15% held out for testing
split = int(len(df) * 0.85)
train, test = df.iloc[:split], df.iloc[split:]
# All engineered features except the raw date and the target ('sales') itself
feature_cols = [c for c in df.columns if c not in ['date', 'sales']]

## Step 3: Train & Evaluate
`GradientBoostingRegressor` builds an ensemble of shallow decision trees sequentially, where each new tree tries to correct the errors of the trees before it — generally a strong performer on tabular feature sets like the one we built. We evaluate with Mean Absolute Error (MAE), the average absolute difference between predicted and actual sales, in the same units as sales itself (so it's directly interpretable, e.g. "off by $12 on average" is easier to reason about than R²). The final plot overlays actual vs predicted sales over the test period: a good model tracks the actual line's ups and downs closely; if you see the predicted line flattening out while actual sales keep climbing, that's the trend-extrapolation limitation of tree-based models mentioned above showing up visually.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

model = GradientBoostingRegressor(random_state=42)
model.fit(train[feature_cols], train['sales'])
preds = model.predict(test[feature_cols])
# MAE: average absolute prediction error, in the same units as sales
print('MAE:', mean_absolute_error(test['sales'], preds))

# Visually compare actual vs predicted sales over the held-out test period
plt.plot(test['date'], test['sales'].values, label='Actual')
plt.plot(test['date'], preds, label='Predicted')
plt.legend()
plt.title('Sales Forecast vs Actual')
plt.show()